<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec06_boundary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec06_boundary.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第6回 境界処理
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec06_boundary.m` を Python に移植したもの。第6回のスライド（vie2026-06）の図と数値を計算する。畳み込みと相互相関，FIR/IIR システム，可分離システム，有限長信号の畳み込みと境界処理（零値・周期・複製・対称拡張），循環畳み込み，内積・ノルムと線形フィルタ，列ベクトル化と行列表現，スペクトルノルムを扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

記号は教科書（村松正吾『多次元信号・画像処理の基礎と展開』）に合わせる。配列の添え字の次元は 1 始まり（$\boldsymbol{n}=(n_1\ n_2)^\top$，$n_1$ が垂直（行），$n_2$ が水平（列））で，各次元の添え字の値は 0 始まり。Python の配列の添字も 0 始まりなので，教科書の添え字の値とそのまま対応する（$n_1$ が軸 0，$n_2$ が軸 1）。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import os
import urllib.request

if not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

from fractions import Fraction

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy import ndimage, signal
from skimage.util import img_as_float
import vie

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

## 二次元畳み込みの計算例
前回スライドの例。入力 $3\times3$，インパルス応答 $3\times3$（中心が原点，$\mathcal{N}_\mathrm{h}=\{-1,0,1\}^2$）。線形畳み込み

$$y[\boldsymbol{n}]=\sum_{\boldsymbol{m}\in\mathcal{N}_x}x[\boldsymbol{m}]h[\boldsymbol{n}-\boldsymbol{m}]$$

の出力は $5\times5$ に広がる（`scipy.signal.convolve2d` の `"full"`）。

In [ ]:
X2 = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
H2 = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
Y2 = signal.convolve2d(X2, H2)                     # full：出力は 5×5
print(Y2)
assert np.array_equal(Y2, [[1, 2, 2, -2, -3], [5, 7, 4, -7, -9], [12, 15, 6, -15, -18],
                           [11, 13, 4, -13, -15], [7, 8, 2, -8, -9]])   # スライドの値

左上の出力 $y[-1,-1]=x[0,0]\,h[-1,-1]$ は 1 項だけの積：

In [ ]:
y00 = Y2[0, 0]
print("y[-1,-1] =", y00)
assert y00 == 1

インパルス応答の重み付け和の一例として，$x[0,0]$ 倍したインパルス応答を位置 $[0\ 0]^\top$ にずらしたもの（出力の左上 $3\times3$）だけを出力の枠に書く。

In [ ]:
Cimp = np.zeros_like(Y2)
Cimp[:3, :3] = X2[0, 0] * H2                       # x[0,0] h[n - [0 0]^T]
print(Cimp)

各軸を反転したインパルス応答（フィルタカーネル）$f[\boldsymbol{n}]=h[-\boldsymbol{n}]$ との相互相関としても同じ結果になる（`scipy.signal.correlate2d` は相関を計算する）。相互相関の例では，カーネルの中心を入力の中央の画素 $x[1,1]$ に合わせる。

In [ ]:
W2 = np.rot90(H2, 2)                               # 各軸を反転
print(W2)
print(np.array_equal(signal.correlate2d(X2, W2, "full"), Y2))

出力の中央 $y[1,1]$（$\mathsf{x}$ の中央の画素に対応）は，反転カーネルを重ねた積和。行の順に項を並べた式も示す。

In [ ]:
def coef2tex(c):
    """係数の表記（負の数は括弧で囲む）"""
    return f"({c:g})" if c < 0 else f"{c:g}"


yc = np.sum(W2 * X2)
print("y[1,1] =", yc)
terms = [coef2tex(w) + "·" + f"{x:d}" for w, x in zip(W2.ravel(), X2.ravel())]   # 行の順（左上→右）
print("+".join(terms), "=", yc)
assert yc == 6 and yc == Y2[2, 2]

## FIR システムと IIR システム
インパルス応答のサポート（非零係数の存在する領域）が有限なら FIR，無限なら IIR。IIR の例として理想低域通過フィルタ（sinc 関数の積）を，FIR の例として $7\times7$ のガウシアンを，同じ範囲に描く。横軸が $n_2$（水平），奥行きが $n_1$（垂直）。

網目の線を高さに応じた色で描く関数と，ガウシアンのカーネルを作る関数を用意しておく。

In [ ]:
def mesh(ax, X, Y, Z, cmap, lw=0.8):
    """網目の線を高さに応じた色で描く（曲面の網目表示）"""
    P = np.stack([X, Y, Z], axis=-1)
    segs = np.concatenate([np.stack([P[:, :-1], P[:, 1:]], axis=2).reshape(-1, 2, 3),
                           np.stack([P[:-1], P[1:]], axis=2).reshape(-1, 2, 3)])
    zc = segs[:, :, 2].mean(axis=1)
    lc = Line3DCollection(segs, cmap=cmap, lw=lw)
    lc.set_array(zc)
    lc.set_clim(Z.min(), Z.max())
    ax.add_collection3d(lc)
    ax.set(xlim=(X.min(), X.max()), ylim=(Y.min(), Y.max()), zlim=(Z.min(), Z.max()))
    return lc


def fspecial_gaussian(K, sigma):
    """K×K のガウシアン exp(-(n1^2+n2^2)/(2σ^2)) を総和 1 に正規化したカーネルを計算する"""
    r = np.arange(K) - (K - 1) / 2
    g = np.exp(-(r[:, None] ** 2 + r[None, :] ** 2) / (2 * sigma ** 2))
    return g / g.sum()

In [ ]:
n2g, n1g = np.meshgrid(np.arange(-15, 16), np.arange(-15, 16))   # 列方向が n2，行方向が n1
hIIR = np.sinc(n1g / 3) * np.sinc(n2g / 3) / 9                   # 無限に広がる（ここでは一部だけ表示）
hFIR = np.zeros(n1g.shape)
g = fspecial_gaussian(7, 1.2)
hFIR[12:19, 12:19] = g                                           # 中央 7×7 以外は零
cmapG = LinearSegmentedColormap.from_list(
    "green", [np.array(cmain) * 0.55 + 0.45, cmain, np.array(cmain) * 0.55])   # 緑の濃淡
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), layout="constrained",
                        subplot_kw={"projection": "3d"})
for ax, h, ttl in zip(axs, (hIIR, hFIR), ("IIR（サポートが無限）", "FIR（サポートが有限）")):
    mesh(ax, n2g, n1g, h, cmapG)
    ax.view_init(elev=35, azim=-120)
    ax.set(title=ttl, xlabel="$n_2$", ylabel="$n_1$")
plt.show()

## 可分離システム
$3\times3$ 矩形フィルタは $h[n_1,n_2]=h_1[n_1]h_2[n_2]$，$h_1=h_2=\frac13(1,1,1)$ と分離できる。二次元畳み込みを一次元畳み込み 2 回で計算しても同じ結果になる。入力には MATLAB 版と同じ $6\times6$ の魔方陣を使う。

In [ ]:
h1 = np.ones((3, 1)) / 3
Hsep = h1 @ h1.T                                   # 1/9 が並ぶ
print(Hsep)
Xr = np.array([[35, 1, 6, 26, 19, 24], [3, 32, 7, 21, 23, 25], [31, 9, 2, 22, 27, 20],
               [8, 28, 33, 17, 10, 15], [30, 5, 34, 12, 14, 16], [4, 36, 29, 13, 18, 11]])   # 6×6 の魔方陣
Ya = signal.convolve2d(Xr, Hsep)                   # 二次元畳み込み
Yb = signal.convolve2d(signal.convolve2d(Xr, h1), h1.T)   # 縦（n1）→横（n2）の一次元畳み込み
maxdiff = np.max(np.abs(Ya - Yb))                  # 丸め誤差程度
print("maxdiff =", maxdiff)

分離可能かどうかは，二次元なら特異値分解で確かめられる（ランク 1 なら非零の特異値は 1 個）。

In [ ]:
print(np.linalg.svd(Hsep, compute_uv=False))

$K\times K$ カーネルの一画素当たりの乗算回数は，そのままだと $K^2$，分離すると $2K$。

In [ ]:
K = np.array([3, 5, 9, 17])
mults = np.vstack([K ** 2, 2 * K])
print(mults)
assert np.array_equal(mults, [[9, 25, 81, 289], [6, 10, 18, 34]])   # スライドの値

## 有限長信号の畳み込み
前回スライドの例。$x[n]=(1,2,4,2)$（$n=0,1,2,3$），$h[n]=(\frac14,\frac12,\frac14)$（$n=-1,0,1$，非因果的）。出力のサポートはミンコフスキー和 $\{0,1,2,3\}+\{-1,0,1\}=\{-1,0,\ldots,4\}$ で，6 点に増える。

In [ ]:
x1 = np.array([1, 2, 4, 2]); nx1 = np.arange(4)
h1d = np.array([1 / 4, 1 / 2, 1 / 4]); nh1 = np.arange(-1, 2)
y1 = np.convolve(x1, h1d)                          # n = -1,0,1,2,3,4
ny1 = np.arange(nx1[0] + nh1[0], nx1[-1] + nh1[-1] + 1)
print("y  =", y1)
print("n  =", ny1)
assert np.array_equal(y1, [0.25, 1, 2.25, 3, 2, 0.5])   # スライドの値

入力・インパルス応答・出力を並べて描く。増えた 2 点（$n=-1,4$）は橙で示す。教科書風の棒グラフ（値のラベルを棒の上に書き，縦軸は描かない）を描く関数を用意しておく。

In [ ]:
def stemtex(ax, n, v, c, labels, xl, yl):
    """教科書風の棒グラフ（stem）．値のラベルを棒の上に書き，縦軸は描かない．
    c は 1 色か，棒ごとの色のリスト．"""
    if np.isscalar(c[0]):
        c = [c] * len(n)
    for nk, vk, ck, lk in zip(n, v, c, labels):
        ax.plot([nk, nk], [0, vk], color=ck, lw=1.6)
        ax.plot(nk, vk, "o", color=ck, ms=5)
        ax.text(nk, vk + 0.25, lk, ha="center", va="bottom", fontsize=9, color=ck)
    ax.set(xlim=xl, ylim=yl, xticks=np.arange(np.ceil(xl[0]), np.floor(xl[1]) + 1))
    ax.spines["bottom"].set_position(("data", 0))
    for s in ("left", "top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_yticks([])
    ax.tick_params(axis="x", direction="inout", labelsize=9)
    ax.patch.set_visible(False)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 2.6), layout="constrained",
                        gridspec_kw={"width_ratios": [3, 2, 3], "wspace": 0.12})
stemtex(axs[0], nx1, x1, ccool, [f"{v:g}" for v in x1], [-1.5, 4.5], [0, 4.8])
axs[0].set_title("$x[n]$")
stemtex(axs[1], nh1, h1d * 4, cmain, ["1/4", "1/2", "1/4"], [-1.8, 1.8], [0, 4.8])   # 見やすさのため 4 倍の高さで描く
axs[1].set_title("$h[n]$")
for xt, s in ((-0.2, "$\\ast$"), (1.2, "$=$")):
    axs[1].text(xt, 0.4, s, transform=axs[1].transAxes, fontsize=16, ha="center", va="center")
stemtex(axs[2], ny1, y1, [cwarm] + [ccool] * 4 + [cwarm], [f"{v:g}" for v in y1], [-1.8, 4.8], [0, 4.8])
axs[2].set_title("$y[n]$")
plt.show()

## 一次元の境界処理
$x[n]=(1,2,4,2)$ を両側 2 点ずつ拡張する。教科書 3.1.3 項の式どおりに外挿の添え字を計算する関数 `bndpad`（次のセル）を使う。NumPy の `np.pad` でも，周期拡張は `"wrap"`，複製拡張は `"edge"`，対称拡張（標本間対称 HS）は `"symmetric"`，標本上対称（WS）は `"reflect"` として得られる。

In [ ]:
def bndpad(X, w, method):
    """教科書 3.1.3 項の境界処理．d = 1,2 の順に各次元を両側 w[d] 点ずつ外挿する．
    method は "zero"（零値），"circ"（周期），"repl"（複製），"hs"（対称 HS），"ws"（対称 WS）．
    外挿幅は N_d - 1 以下を想定する（教科書の式と同じ）．一次元の配列は 1 行の配列として扱う．"""
    Xp = np.atleast_2d(np.asarray(X, dtype=np.float64))
    for d in range(2):
        N = Xp.shape[d]
        n = np.arange(-w[d], N + w[d])                 # 拡張後の添え字（0 始まり）
        m = n.astype(np.float64)                       # 参照する元の添え字（NaN は零値）
        lo, hi = n < 0, n > N - 1
        if method == "zero":
            m[lo | hi] = np.nan
        elif method == "circ":
            m = np.mod(n, N).astype(np.float64)
        elif method == "repl":
            m[lo], m[hi] = 0, N - 1
        elif method == "hs":
            m[lo], m[hi] = -n[lo] - 1, 2 * N - 1 - n[hi]
        elif method == "ws":
            m[lo], m[hi] = -n[lo], 2 * (N - 1) - n[hi]
        ok = ~np.isnan(m)
        shape = list(Xp.shape)
        shape[d] = n.size
        Y = np.zeros(shape)
        if d == 0:
            Y[ok, :] = Xp[m[ok].astype(int), :]
        else:
            Y[:, ok] = Xp[:, m[ok].astype(int)]
        Xp = Y
    return Xp


methods = ["zero", "circ", "repl", "hs", "ws"]
npmodes = ["constant", "wrap", "edge", "symmetric", "reflect"]   # np.pad の対応するモード

In [ ]:
pads = np.vstack([bndpad(x1, (0, 2), m)[0] for m in methods])
print(pads)
print(np.array_equal(pads, np.vstack([np.pad(x1, 2, mode=md) for md in npmodes])))
assert np.array_equal(pads, [[0, 0, 1, 2, 4, 2, 0, 0], [4, 2, 1, 2, 4, 2, 1, 2],
                             [1, 1, 1, 2, 4, 2, 2, 2], [2, 1, 1, 2, 4, 2, 2, 4],
                             [4, 2, 1, 2, 4, 2, 4, 2]])   # スライドの値

## 画像の境界処理の例
教科書のサンプル画像 msipimg06（縞模様の路面，256×256 に縮小）に $17\times17$ のガウシアン（$\sigma_\mathrm{g}=4$）を施す。拡張点数は 16 点（両側 8 点）。路面が明るく，画像の四辺とも明るい画素が接しているので，零値拡張で縁が黒く滲む様子がどの辺でもはっきり見える。対称拡張では滲まない。

フィルタリングは `scipy.ndimage.correlate`（相関）で計算する。零値拡張は `mode="constant"`，対称拡張は `mode="reflect"`。カラー画像は色ごとに同じカーネルを施す。

In [ ]:
Xc = img_as_float(vie.msipimg(6, 256))            # 縞模様の路面（256×256）
Xc = np.clip(Xc, 0, 1)
fgau = fspecial_gaussian(17, 4)
Ez = np.pad(Xc, ((8, 8), (8, 8), (0, 0)))                       # 零値拡張画像
Es = np.pad(Xc, ((8, 8), (8, 8), (0, 0)), mode="symmetric")     # 対称拡張画像
Yz = ndimage.correlate(Xc, fgau[:, :, None], mode="constant", cval=0)   # 零値拡張で処理
Ys = ndimage.correlate(Xc, fgau[:, :, None], mode="reflect")            # 対称拡張で処理

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8, 8), layout="constrained")
show(axs[0, 0], Ez, "零値拡張画像")
show(axs[0, 1], Yz, "処理画像（零値拡張）")
show(axs[1, 0], Es, "対称拡張画像")
show(axs[1, 1], Ys, "処理画像（対称拡張）")
plt.show()

左上隅の画素の明るさ（R 成分）を比べると，零値拡張では暗くなる（縮小の実装が MATLAB と異なるため，値はスライドとわずかに異なることがある）。

In [ ]:
cornerv = np.array([Xc[0, 0, 0], Yz[0, 0, 0], Ys[0, 0, 0]])
print("元画像 {:.2f}，零値拡張 {:.2f}，対称拡張 {:.2f}".format(*cornerv))

## 例題「境界処理」（教科書 3.1.3 項）
例題「矩形フィルタ」（3.1.1 項）の配列

$$\mathsf{x}=\begin{pmatrix}18&9&9&9\\27&9&9&9\\36&9&9&9\end{pmatrix}\in\mathbb{R}^{3\times4}$$

に，零値拡張，周期拡張，複製拡張，対称拡張（HS・WS）を施す。ただし $\mathcal{N}_\mathrm{pad}=\{-2,-1,\ldots,4\}\times\{-2,-1,\ldots,5\}$（両側 2 点ずつ，$7\times8$）。`bndpad` は教科書の式どおり $d=1,2$ の順に各次元を外挿する。

In [ ]:
Xb = np.array([[18, 9, 9, 9], [27, 9, 9, 9], [36, 9, 9, 9]])
Pb = [bndpad(Xb, (2, 2), m) for m in methods]
for m, P in zip(methods, Pb):
    print(m)
    print(P.astype(int))

`np.pad` の結果と一致することを確かめる。

In [ ]:
print(all(np.array_equal(P, np.pad(Xb, 2, mode=md)) for P, md in zip(Pb, npmodes)))

教科書の解答と照合する（解答の配列をそのまま書き写したもの）。

In [ ]:
Tb = [
    [[0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 18, 9, 9, 9, 0, 0], [0, 0, 27, 9, 9, 9, 0, 0],
     [0, 0, 36, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0]],
    [[9, 9, 27, 9, 9, 9, 27, 9], [9, 9, 36, 9, 9, 9, 36, 9], [9, 9, 18, 9, 9, 9, 18, 9], [9, 9, 27, 9, 9, 9, 27, 9],
     [9, 9, 36, 9, 9, 9, 36, 9], [9, 9, 18, 9, 9, 9, 18, 9], [9, 9, 27, 9, 9, 9, 27, 9]],
    [[18, 18, 18, 9, 9, 9, 9, 9], [18, 18, 18, 9, 9, 9, 9, 9], [18, 18, 18, 9, 9, 9, 9, 9], [27, 27, 27, 9, 9, 9, 9, 9],
     [36, 36, 36, 9, 9, 9, 9, 9], [36, 36, 36, 9, 9, 9, 9, 9], [36, 36, 36, 9, 9, 9, 9, 9]],
    [[9, 27, 27, 9, 9, 9, 9, 9], [9, 18, 18, 9, 9, 9, 9, 9], [9, 18, 18, 9, 9, 9, 9, 9], [9, 27, 27, 9, 9, 9, 9, 9],
     [9, 36, 36, 9, 9, 9, 9, 9], [9, 36, 36, 9, 9, 9, 9, 9], [9, 27, 27, 9, 9, 9, 9, 9]],
    [[9, 9, 36, 9, 9, 9, 9, 9], [9, 9, 27, 9, 9, 9, 9, 9], [9, 9, 18, 9, 9, 9, 9, 9], [9, 9, 27, 9, 9, 9, 9, 9],
     [9, 9, 36, 9, 9, 9, 9, 9], [9, 9, 27, 9, 9, 9, 9, 9], [9, 9, 18, 9, 9, 9, 9, 9]],
]
matchText = [np.array_equal(P, T) for P, T in zip(Pb, Tb)]   # すべて True なら教科書と一致
print(matchText)
assert all(matchText), "例題「境界処理」の結果が教科書の解答と一致しない"

元の配列 $\mathsf{x}$ の位置（$\mathcal{N}_x=\{0,1,2\}\times\{0,1,2,3\}$）の要素は，拡張後の配列の行 2〜4，列 2〜5 にある。

In [ ]:
isOrg = np.zeros((7, 8), dtype=bool)
isOrg[2:5, 2:6] = True
print(all(np.array_equal(P[isOrg].reshape(3, 4), Xb) for P in Pb))

## 例題：対称拡張（WS）による有限長信号のフィルタリング
WS で拡張してから畳み込み，元の 4 点だけを取り出す（棄却・クリッピング）。出力の長さは入力と同じ 4 点。

In [ ]:
xW = bndpad(x1, (0, 1), "ws")[0]                   # n = -1,0,...,4（両側 1 点で足りる）
yW = np.convolve(xW, h1d, "valid")                 # n = 0,1,2,3
yZ = np.convolve(x1, h1d, "same")                  # 零値拡張の場合（端が小さくなる）
print("xW =", xW)
print("yW =", yW)
print("yZ =", yZ)
assert np.array_equal(yW, [1.5, 2.25, 3, 3]) and np.array_equal(yZ, [1, 2.25, 3, 2])   # スライドの値

両端の出力 $y[0]$ と $y[3]$ の計算式 $y[n]=\sum_{m=-1}^{1}h[m]\,x_\mathrm{pad}[n-m]$（$m=-1,0,1$ の順）も示す。

In [ ]:
def frac2str(v):
    """有理数を p/q の形にする（整数ならそのまま）"""
    f = Fraction(v).limit_denominator(1000)
    return f"{f.numerator}" if f.denominator == 1 else f"{f.numerator}/{f.denominator}"


for n in (0, 3):
    terms = [frac2str(h1d[m + 1]) + "·" + f"{xW[n - m + 1]:g}" for m in (-1, 0, 1)]   # xW[0] が n=-1
    print(f"y[{n}] =", "+".join(terms), "=", f"{yW[n]:g}")

図：WS 拡張した信号（灰色が拡張した点），インパルス応答，出力。前回演習課題（6）の図（対称の中心の矢印）を参考にした。

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 2.8), layout="constrained",
                        gridspec_kw={"width_ratios": [3, 2, 3], "wspace": 0.12})
stemtex(axs[0], np.arange(-1, 5), xW, [cgray] + [ccool] * 4 + [cgray], [f"{v:g}" for v in xW],
        [-1.8, 4.8], [-2.1, 4.8])
for nc in (0, 3):
    axs[0].annotate("", xy=(nc, -1.25), xytext=(nc, -2.1),
                    arrowprops=dict(arrowstyle="-|>", color=cwarm, lw=1.2))
axs[0].text(1.5, -1.7, "対称の中心", ha="center", fontsize=8, color=cwarm)
axs[0].set_title("$x_{\\mathrm{pad}}[n]$")
stemtex(axs[1], nh1, h1d * 4, cmain, ["1/4", "1/2", "1/4"], [-1.8, 1.8], [-2.1, 4.8])
axs[1].set_title("$h[n]$")
for xt, s in ((-0.2, "$\\ast$"), (1.2, "$=$")):
    axs[1].text(xt, 0.5, s, transform=axs[1].transAxes, fontsize=16, ha="center", va="center")
stemtex(axs[2], np.arange(4), yW, ccool, [f"{v:g}" for v in yW], [-1.8, 4.8], [-2.1, 4.8])
axs[2].set_title("$y[n]$")
plt.show()

## 例題「インパルス応答の周期化」と「循環畳み込みによる線形畳み込み」（教科書 4.3.2・4.3.3 項）
$3\times3$ 矩形フィルタのインパルス応答 $\mathsf{h}$（$\mathcal{N}_\mathrm{h}=\{-1,0,1\}^2$）を周期構造行列 $\boldsymbol{N}=\mathrm{diag}(6,8)$ で周期化する：

$$h_{\boldsymbol{N}}[\boldsymbol{n}]=\sum_{\boldsymbol{\ell}\in\mathbb{Z}^2}h[\boldsymbol{N}\boldsymbol{\ell}+\boldsymbol{n}],\quad\boldsymbol{n}\in\mathcal{N}(\boldsymbol{N})=\{0,\ldots,5\}\times\{0,\ldots,7\}$$

負の添え字の係数が反対側に回り込む。

In [ ]:
Nper = (6, 8)
hN = np.zeros(Nper)
for m1 in range(-1, 2):
    for m2 in range(-1, 2):
        hN[m1 % Nper[0], m2 % Nper[1]] += 1 / 9
hN9 = np.round(9 * hN).astype(int)               # 1/9 をくくり出した整数の配列
print(hN9)
Tper = np.array([[1, 1, 0, 0, 0, 0, 0, 1], [1, 1, 0, 0, 0, 0, 0, 1], [0] * 8, [0] * 8, [0] * 8,
                 [1, 1, 0, 0, 0, 0, 0, 1]])        # 教科書の解答（×9）
assert np.array_equal(hN9, Tper), "例題「インパルス応答の周期化」の結果が教科書と一致しない"

例題「矩形フィルタ」の $\mathsf{x}$（$\mathcal{N}(\boldsymbol{N})$ の残りは零）との循環畳み込み

$$y[\boldsymbol{n}]=\sum_{\boldsymbol{m}\in\mathcal{N}(\boldsymbol{N})}h_{\boldsymbol{N}}[\boldsymbol{m}]\,x[((\boldsymbol{n}-\boldsymbol{m}))_{\boldsymbol{N}}]$$

を定義どおりに計算し，DFT の積（$\tilde{\mathsf{y}}=\tilde{\mathsf{h}}\odot\tilde{\mathsf{x}}$，`np.fft.fft2`）による結果とも比べる。

In [ ]:
XN = np.zeros(Nper)
XN[:3, :4] = Xb
Ycc = np.zeros(Nper)
for n1 in range(Nper[0]):
    for n2 in range(Nper[1]):
        for m1 in range(Nper[0]):
            for m2 in range(Nper[1]):
                Ycc[n1, n2] += hN[m1, m2] * XN[(n1 - m1) % Nper[0], (n2 - m2) % Nper[1]]
Ycc = np.round(Ycc, 10)                            # 小数点以下 10 桁で丸める
print(Ycc.astype(int))
Yfft = np.real(np.fft.ifft2(np.fft.fft2(hN) * np.fft.fft2(XN)))
maxdiffFFT = np.max(np.abs(Ycc - Yfft))            # 丸め誤差程度
print("maxdiffFFT =", maxdiffFFT)
Tcc = np.array([[7, 9, 6, 4, 2, 0, 0, 5], [12, 15, 9, 6, 3, 0, 0, 9], [9, 11, 6, 4, 2, 0, 0, 7],
                [5, 6, 3, 2, 1, 0, 0, 4], [0, 0, 0, 0, 0, 0, 0, 0], [3, 4, 3, 2, 1, 0, 0, 2]])   # 教科書の解答
assert np.array_equal(Ycc, Tcc), "例題「循環畳み込みによる線形畳み込み」の結果が教科書と一致しない"

左上の $3\times4$ は例題「矩形フィルタ」（零値拡張の線形フィルタ）の結果と一致する。

In [ ]:
Ybox = signal.convolve2d(Xb, np.ones((3, 3)) / 9, "same")
print(Ybox)
print(np.array_equal(Ycc[:3, :4], np.round(Ybox, 10)))

## 内積・ノルム・距離・コサイン類似度
教科書 1.2.1 項の例。二つの配列の内積，ノルム，距離，コサイン類似度，二乗誤差和。教科書の値（80，10，16，14，1/2，196）と一致する。

In [ ]:
Xa = np.array([[7, 5, 3], [3, 2, 2]]); Ya6 = np.array([[-4, 9, 6], [7, 5, 7]])
ip = np.sum(Xa * Ya6)
nx, ny = np.linalg.norm(Xa.ravel()), np.linalg.norm(Ya6.ravel())
dist = np.linalg.norm(Xa.ravel() - Ya6.ravel())
cosv = ip / (nx * ny)
sse = dist ** 2
l1 = np.linalg.norm(Xa.ravel(), 1)                 # 1-ノルム（絶対値和）
for tag, v in zip(["ip", "nx", "ny", "dist", "cos", "sse", "l1"], [ip, nx, ny, dist, cosv, sse, l1]):
    print(f"{tag:4s} = {v:g}")
assert (ip, nx, ny, dist, cosv, sse, l1) == (80, 10, 16, 14, 0.5, 196, 22)   # スライドの値

コサイン類似度を分数で，二つの配列のなす角 $\theta=\arccos(1/2)$ を $\pi$ の分数で示す。

In [ ]:
theta = np.arccos(cosv)
print("θ =", theta)
ft = Fraction(theta / np.pi).limit_denominator(1000)   # θ/π = tn/td
snum = "" if ft.numerator == 1 else f"{ft.numerator}"
print("cos =", frac2str(cosv), "，θ =", snum + "π/" + f"{ft.denominator}")
assert frac2str(cosv) == "1/2" and (ft.numerator, ft.denominator) == (1, 3)

## 移動平均・移動差分は内積
$\mathbb{R}^2$ で平均 $\mathbf{f}=\frac12(1\ 1)^\top$，差分 $\mathbf{f}=(1\ -1)^\top$ と $\mathbf{v}$ との内積 $\langle\mathbf{f},\mathbf{v}\rangle$ をとる。内積が 0 なら直交。

In [ ]:
V = np.array([[1, 1], [1, -1], [2, 1]]).T          # 列が v
avg2 = np.array([1 / 2, 1 / 2]) @ V
dif2 = np.array([1, -1]) @ V
print("平均:", avg2)
print("差分:", dif2)
assert np.array_equal(avg2, [1, 0, 1.5]) and np.array_equal(dif2, [0, 2, 1])   # スライドの値

$\mathbb{R}^{2\times2}$ の配列 $\mathsf{x}$ との内積（成分毎の積の和）で，平均・水平差分・垂直差分を抽出する。

In [ ]:
Vv = np.array([[2, 2], [1, 1]])
Favg = np.ones((2, 2)) / 4; Fh = np.array([[-1, 1], [-1, 1]]); Fv = np.array([[-1, -1], [1, 1]])
ext = np.array([np.sum(Favg * Vv), np.sum(Fh * Vv), np.sum(Fv * Vv)])
print(ext)
assert np.array_equal(ext, [1.5, 0, -2])           # スライドの値

## 画像フィルタ
各画素の近傍 $3\times3$ 配列とフィルタカーネルの内積を全画素で計算する。教科書のサンプル画像 msipimg04（石造りの建物，グレースケール，256×256 に縮小）に矩形フィルタ，4 近傍ラプラシアン，ソーベル（水平・垂直）を施す（境界は対称拡張。負の値を含む出力は 0.5 を中心に表示）。建物は柱（縦）と階段・軒（横）の輪郭が多いので，水平ソーベルが縦の輪郭を，垂直ソーベルが横の輪郭を取り出す違いがはっきり見える。

In [ ]:
Cm = img_as_float(vie.msipimg(4, 256, "gray"))   # 石造りの建物（256×256）
masks = [np.ones((3, 3)) / 9, np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
         np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]), np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])]
mttl = ["矩形フィルタ", "4 近傍ラプラシアン", "ソーベル（水平）", "ソーベル（垂直）"]
fig, axs = plt.subplots(1, 4, figsize=(12, 3.4), layout="constrained")
for k, (ax, mk) in enumerate(zip(axs, masks)):
    Yk = ndimage.correlate(Cm, mk, mode="reflect")   # correlate は相関（カーネルをそのまま重ねる）
    if k > 0:
        Yk = 0.5 + Yk / (2 * np.max(np.abs(Yk)))   # 負の値を含むので 0.5 を中心に表示
    show(ax, Yk, mttl[k])
plt.show()

## 配列の列ベクトル化
列の順（縦方向 $n_1$ が先）に並べる。NumPy の既定は行の順なので `order="F"` を指定する。$2\times2$ の例：

In [ ]:
Xv = np.array([[0, 2], [4, 6]])
xv = Xv.ravel(order="F")
Xback = xv.reshape((2, 2), order="F")              # 配列化（逆写像）
print("xv =", xv)
print(Xback)
assert np.array_equal(xv, [0, 4, 2, 6]) and np.array_equal(Xback, Xv)   # スライドの値

4K 画像（$2160\times3840$ 画素）は約 800 万次元のベクトル：

In [ ]:
dim4k = 2160 * 3840
print(f"{dim4k:,}")
assert dim4k == 8294400

## 例「線形写像と行列表現」：循環右シフト（教科書 1.2.2 項）
$\mathbb{R}^{2\times3}$ の配列を列ごとに右へ循環シフトする写像 $\mathsf{T}$ は線形で，行列表現 $\mathbf{T}\in\mathbb{R}^{6\times6}$ をもつ。$\mathbf{T}$ の第 $i$ 列は，標準基底 $\{\delta[\boldsymbol{n}-\boldsymbol{m}_i]\}_{\boldsymbol{n}}$ に対する出力を列ベクトル化したもの。

In [ ]:
Tsh = np.zeros((6, 6), dtype=int)
for i in range(6):
    E = np.zeros(6, dtype=int); E[i] = 1
    E = E.reshape((2, 3), order="F")               # 標準基底（列ベクトル化の順に i 番目）
    R = np.roll(E, 1, axis=1)                      # 循環右シフト
    Tsh[:, i] = R.ravel(order="F")
print(Tsh)
Ttext = np.array([[0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0],
                  [0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0]])   # 教科書
assert np.array_equal(Tsh, Ttext), "例「線形写像と行列表現」の行列が教科書と一致しない"

例「スペクトルノルム」：スペクトルノルム $\|\mathsf{T}\|_\mathrm{S}=\sqrt{\lambda_0(\mathbf{T}^\top\mathbf{T})}$。循環シフトはノルムを保存する（等長）ので 1。

In [ ]:
snShift = np.sqrt(np.max(np.abs(np.linalg.eigvals(Tsh.T @ Tsh))))
print("snShift =", snShift)
print("2-ノルム（最大特異値）:", np.linalg.norm(Tsh, 2))   # 2-ノルム（最大特異値）でも同じ
assert np.isclose(snShift, 1)

## 対称畳み込みの行列表現
前回スライドの例。$2\times3$ 配列 $\mathsf{x}$ を WS 対称拡張（周期的にも延長）して $\mathsf{x}_\mathrm{pad}$ とし，インパルス応答 $h[\boldsymbol{m}]$（$\boldsymbol{m}\in\{0,1\}^2$，$h[0,0]=h[1,0]=1,\ h[0,1]=h[1,1]=-1$）と畳み込んで $\mathsf{y}$ を得る。

In [ ]:
def wsidx(n, N):
    """WS 対称＋周期の添え字（0 始まり）"""
    return N - 1 - np.abs(np.mod(n, 2 * (N - 1)) - (N - 1))


def symconv(X, H):
    """WS 対称拡張（周期的にも延長）した X とインパルス応答 H（添え字 {0,1}^2）の畳み込み"""
    N1, N2 = X.shape
    Y = np.zeros((N1, N2))
    for a in range(N1):
        for b in range(N2):
            for m1 in range(H.shape[0]):
                for m2 in range(H.shape[1]):
                    Y[a, b] += H[m1, m2] * X[wsidx(a - m1, N1), wsidx(b - m2, N2)]
    return Y


U = np.array([[0, 2, 4], [1, 3, 5]])               # 入力 x
Hk = np.array([[1, -1], [1, -1]])                  # 行が m1，列が m2
Vout = symconv(U, Hk)                              # 出力 y
print(Vout.astype(int))
Uext = np.array([[U[wsidx(i, 2), wsidx(j, 3)] for j in range(-1, 4)] for i in range(-1, 3)])   # 拡張の様子
print(Uext)
assert np.array_equal(Vout, [[-4, 4, 4], [-4, 4, 4]])   # スライドの値
assert np.array_equal(Uext, [[3, 1, 3, 5, 3], [2, 0, 2, 4, 2], [3, 1, 3, 5, 3], [2, 0, 2, 4, 2]])

このシステムは線形なので行列表現 $\mathbf{T}$ をもつ（$\mathbf{y}=\mathbf{T}\mathbf{x}$）。$\mathbf{T}$ の各列は，標準基底に対する出力を列ベクトル化したもの。

In [ ]:
T = np.zeros((6, 6))
for i in range(6):
    E = np.zeros(6); E[i] = 1
    Vm = symconv(E.reshape((2, 3), order="F"), Hk)
    T[:, i] = Vm.ravel(order="F")
print(T.astype(int))
check = np.array_equal(T @ U.ravel(order="F"), Vout.ravel(order="F"))   # y = T x
print("check:", check)
assert check and np.array_equal(T, [[1, 1, -1, -1, 0, 0], [1, 1, -1, -1, 0, 0], [-1, -1, 1, 1, 0, 0],
                                    [-1, -1, 1, 1, 0, 0], [0, 0, -1, -1, 1, 1], [0, 0, -1, -1, 1, 1]])

## 例題「矩形フィルタのスペクトルノルム」（教科書 4.3.3 項）
循環畳み込み $\mathsf{T}(\cdot)=\mathsf{h}_{\boldsymbol{N}}\circledast(\cdot)$ の行列表現は $\mathbf{T}=\mathbf{W}_{\boldsymbol{N}}^{-1}\boldsymbol{\Lambda}_\mathrm{h}\mathbf{W}_{\boldsymbol{N}}$（$\mathbf{W}_{\boldsymbol{N}}$ は DFT 行列，$\boldsymbol{\Lambda}_\mathrm{h}=\mathrm{diag}(\mathbf{W}_{\boldsymbol{N}}\mathbf{h})$）なので，スペクトルノルムは DFT 係数の絶対値の最大値

$$\|\mathsf{T}\|_\mathrm{S}=\max_{\boldsymbol{k}\in\mathcal{N}(\boldsymbol{N}^\top)}|H[\boldsymbol{k}]|$$

になる。$3\times3$ 矩形フィルタ，$\boldsymbol{N}=\mathrm{diag}(6,8)$ で確かめる。

In [ ]:
Hk68 = np.fft.fft2(hN)                             # H[k]，k ∈ {0,...,5}×{0,...,7}
snBox = np.max(np.abs(Hk68))
kmax = tuple(int(k) for k in np.unravel_index(np.argmax(np.abs(Hk68)), Nper))
print("snBox =", snBox, "，kmax =", kmax)          # 最大は k = 0（直流）

行列表現 $\mathbf{T}\in\mathbb{R}^{48\times48}$ を標準基底に対する応答から作り，DFT 行列による対角化と最大特異値でも確かめる。列ベクトル化は列の順なので $\mathbf{W}_{\boldsymbol{N}}=\mathbf{W}_8\otimes\mathbf{W}_6$。

In [ ]:
P48 = Nper[0] * Nper[1]
Tcc48 = np.zeros((P48, P48))
for i in range(P48):
    E = np.zeros(P48); E[i] = 1
    R = np.real(np.fft.ifft2(np.fft.fft2(hN) * np.fft.fft2(E.reshape(Nper, order="F"))))
    Tcc48[:, i] = R.ravel(order="F")
W6 = np.exp(-2j * np.pi * np.outer(np.arange(6), np.arange(6)) / 6)
W8 = np.exp(-2j * np.pi * np.outer(np.arange(8), np.arange(8)) / 8)
WN = np.kron(W8, W6)
maxdiffDiag = np.max(np.abs(Tcc48 - np.linalg.solve(WN, np.diag(WN @ hN.ravel(order="F")) @ WN)))   # 丸め誤差程度
snBox48 = np.linalg.norm(Tcc48, 2)                 # 最大特異値 = スペクトルノルム
print("maxdiffDiag =", maxdiffDiag)
print("snBox48 =", snBox48)
assert abs(snBox - 1) < 1e-12 and abs(snBox48 - 1) < 1e-12, "スペクトルノルムが教科書の値 1 と一致しない"

振幅応答 $|H(\mathrm{e}^{\mathrm{j}\boldsymbol{\omega}^\top})|=\frac19|1+2\cos\omega_1|\cdot|1+2\cos\omega_2|$ と，DFT の周波数標本点 $\boldsymbol{\omega}=2\pi\boldsymbol{N}^{-\top}\boldsymbol{k}$ を重ねて描く（ライブスクリプトでの確認用で，スライドには載せない）。標本の最大は $\boldsymbol{k}=\mathbf{0}$ の 1。

In [ ]:
w = np.linspace(0, 2 * np.pi, 241)
w2g, w1g = np.meshgrid(w, w)                       # 列方向が ω2，行方向が ω1
Hmag = np.abs(1 + 2 * np.cos(w1g)) * np.abs(1 + 2 * np.cos(w2g)) / 9
k2g, k1g = np.meshgrid(np.arange(Nper[1]), np.arange(Nper[0]))
w1k = 2 * np.pi * k1g / Nper[0]; w2k = 2 * np.pi * k2g / Nper[1]
print(np.max(np.abs(np.abs(Hk68) - np.abs(1 + 2 * np.cos(w1k)) * np.abs(1 + 2 * np.cos(w2k)) / 9)))   # 標本点で一致

In [ ]:
cmapW = LinearSegmentedColormap.from_list("white_green", [(1, 1, 1), cmain])   # 白→緑
fig, ax = plt.subplots(figsize=(5.6, 4.6), layout="constrained")
im = ax.imshow(Hmag, extent=(w[0], w[-1], w[0], w[-1]), origin="lower", cmap=cmapW, vmin=0, vmax=1)
fig.colorbar(im, ax=ax)
ax.plot(w2k.ravel(), w1k.ravel(), ".", color=ccool, ms=8)
ax.plot(0, 0, "o", mfc="none", color=cwarm, ms=10, mew=1.5)
ax.set(xlim=(-0.45, 2 * np.pi + 0.1), ylim=(-0.45, 2 * np.pi + 0.1),   # 原点の丸が隠れないよう余白をとる
       xticks=[0, np.pi, 2 * np.pi], yticks=[0, np.pi, 2 * np.pi],
       xticklabels=["$0$", "$\\pi$", "$2\\pi$"], yticklabels=["$0$", "$\\pi$", "$2\\pi$"],
       xlabel="$\\omega_2$", ylabel="$\\omega_1$",
       title="$|H(\\mathrm{e}^{\\mathrm{j}\\boldsymbol{\\omega}^\\top})|$ と DFT の標本点（橙：最大）")
ax.tick_params(direction="out")
plt.show()

## まとめ
- 畳み込みは反転したインパルス応答（フィルタカーネル）との相互相関（荷重移動平均／差分）に一致する
- 可分離システムは一次元処理の繰り返しで計算でき，演算量が少ない
- 有限長信号の畳み込みは出力が広がるので，境界処理（零値・周期・複製・対称拡張）が必要
- 周期拡張と畳み込みは循環畳み込みで，十分な周期をとれば線形畳み込みを実現できる
- 線形フィルタは内積の繰り返しであり，列ベクトル化により行列 $\mathbf{T}$ で表現できる（$\mathbf{y}=\mathbf{T}\mathbf{x}$）。作用の大きさはスペクトルノルムで測る

© Copyright, Shogo MURAMATSU, All rights reserved.